## Using `RailProject` 

This notebook will show you the basics using the `RailProject` class to manage an analysis project

### Setup and teardown scripts to setup a test area

In [ ]:
import os
#from rail.projects import library
#
check_dir = os.path.basename(os.path.abspath(os.curdir))
if check_dir == 'examples':
    os.chdir('..')

#setup = library.setup_project_area()
#assert setup == 0

# use this to cleanup
# library.teardown_project_area()

### Load the test project

In [ ]:
from rail.projects import RailProject

project = RailProject.load_config("./examples/cardinal_project.yaml")

### Inspect the test project

In [ ]:
catalog_files_truth = project.get_catalog_files("truth")
print(catalog_files_truth)

In [ ]:
list(catalog_files_truth)

### Run a data reduction algorithm on the test project data

This will use the "roman_rubin" reducer to apply the "gold" selection to the "truth" catalog to make a "reduced" catalog

In [ ]:
project.reduce_data(
    catalog_template="truth",
    output_catalog_template="reduced",
    reducer_class_name="cardinal",
    input_selection="",
    selection="gold",
)


### Subsample the test project

This will use the "random_subsampler" to apply the "train_10" subsample to the "reduced" catalog of the baseline flavor with the gold selection

In [ ]:
project.subsample_data(
    catalog_template="reduced",
    file_template="train_file_200k",
    subsampler_class_name="random_subsampler",
    subsample_name="train_200k",
    flavor="baseline",
    selection="gold",
)

In [ ]:
import tables_io

In [ ]:
goldfile = "/global/cfs/cdirs/lsst/groups/PZ/Cardinal/parquet_files/test/test_cardinal_gold_baseline_200k.parquet"
goldtab = tables_io.read(goldfile)

In [ ]:
golddf = tables_io.convert(goldtab, tables_io.types.PD_DATAFRAME)

In [ ]:
golddf.info()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(golddf['ra'], golddf['dec'], s=1, c='k')

In [ ]:
gr = golddf['mag_g_lsst'] - golddf['mag_r_lsst']

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(golddf['redshift'], gr, s=1,c='k')
plt.xlabel("redshift")
plt.ylabel("g-r")

In [ ]:
# difference of cosmological redshift (w proper motion) - true z vs true redshift
delz = golddf['redshift'] - golddf['true_redshift']
plt.figure(figsize=(14,8))
plt.scatter(golddf['redshift'], delz, s=.1, c='k')
plt.xlabel("true redshift")
plt.ylabel("z cosmological - z true")

In [ ]:
# plot the redshift distribution of the "gold" i<25.5 sample:
import numpy as np
plt.figure(figsize=(10,6))
plt.hist(golddf['redshift'], bins=np.linspace(0,2.5,101), color='k');
plt.xlabel("redshift")
plt.ylabel("Number")